Author: Daniel Lillard

Date: 2026.03.23

Desc: Basic LSTM model here to predict stock prices on the smoothed data. This is a "worst possible" algorithm incase the clustering is not useful.

In [28]:
def convert_and_cut_date(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df[df['timestamp'] >= start_date]
    df = df[df['timestamp'] <= end_date]
    return df

In [106]:
# smooth all data

def smooth_data(df: pd.DataFrame) -> pd.DataFrame:
    # smooth the data using a rolling window of 5 days
    raw_close = df.pivot(columns="ticker", values="close",index="timestamp")
    smoothed  = raw_close.rolling(window=5, min_periods=1).mean()

    
    
    return smoothed

In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def build_model(window: int) -> Sequential:
    # CNN-LSTM architecture from the paper.
    model = Sequential([
        Conv1D(filters=5, kernel_size=3, activation="relu",
               padding="same", input_shape=(window, 1)),
        Dropout(0.2),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(60, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

In [107]:
# MAIN
import pandas as pd

TRAIN_START  = "2019-07-01"
TRAIN_END    = "2021-12-31"   # last training day before test window
TEST_START   = "2022-01-03"
TEST_END     = "2022-03-31"
 
WINDOW_SIZE  = 30             # number of past days used as input
EPOCHS       = 20
BATCH_SIZE   = 32

df = pd.read_csv("aa_daily_data.csv")

# for testing, only use 5 stocks
# df = df[df['ticker'].isin(['ACHC', 'AEE', 'HCKT', 'HALO', 'MA'])]

# for testing, only use 1 stocks
df = df[df['ticker'].isin(['HALO'])]

df = convert_and_cut_date(df, TRAIN_START, TEST_END)

df_smoothed = smooth_data(df)

raw_open = df.pivot(columns='ticker',index='timestamp')['open']
raw_close = df.pivot(columns='ticker',index='timestamp')['close']

# # lets start with just one day to make sure the pipeline works
test_date = "2022-01-03"
# test_data = df[df['timestamp'] == test_date]

# train_data = df_smoothed[df_smoothed['timestamp'] < test_date]
# # only go back window size days from the test date to avoid data leakage
# train_data = train_data[train_data['timestamp'] >= (pd.to_datetime(test_date) - pd.Timedelta(days=WINDOW_SIZE))]


early_stop = EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)

model = build_model(WINDOW_SIZE)

c:\file_struct\programs\python\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [111]:
# Scale
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df_smoothed.values.reshape(-1, 1)).flatten()

In [112]:
import numpy as np

def build_sequences(series: np.ndarray, window: int):
    """
    Build (X, y) pairs from a 1D scaled series.
    X shape: (n_samples, window, 1)
    y shape: (n_samples)
    """
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i - window:i])
        y.append(series[i])
    return np.array(X), np.array(y)

In [113]:
X, y = build_sequences(scaled, WINDOW_SIZE)
X    = X[..., np.newaxis]   # (n, window, 1)

In [114]:
tf.keras.backend.clear_session()
model = build_model(WINDOW_SIZE)
model.fit(X, y,
          epochs=EPOCHS,
          batch_size=BATCH_SIZE,
          callbacks=[early_stop],
          verbose=0)

# Predict
last_window = scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
pred_scaled = model.predict(last_window, verbose=0)[0, 0]
pred_price  = scaler.inverse_transform([[pred_scaled]])[0, 0]

c:\file_struct\programs\python\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [115]:
opens = raw_open['HALO']

In [ ]:
pred_price, opens.get(test_date, default=np.nan)

(np.float64(38.36521048879623), np.float64(39.8))

In [119]:
# ─────────────────────────────────────────────
# TRADING SIGNALS
# ─────────────────────────────────────────────
 
def trading_signal(prev_close: float, open_t: float, predicted: float) -> str:
    """
    Buy  if prev_close < predicted AND open_t < predicted
    Sell if prev_close > predicted AND open_t > predicted
    """
    if prev_close < predicted and open_t < predicted:
        return "BUY"
    elif prev_close > predicted and open_t > predicted:
        return "SELL"
    return "HOLD"
 
 
def compute_trade_return(signal: str, open_t: float, close_t: float) -> float:
    if signal == "BUY":
        return (close_t - open_t) / open_t
    elif signal == "SELL":
        return (open_t - close_t) / open_t
    return 0.0

In [134]:
test_date_small = "2022-01-03"

raw_close.loc[test_date_small]

ticker
HALO    41.06
Name: 2022-01-03 00:00:00, dtype: float64

In [139]:
ticker = "HALO"

actual_close = raw_close.loc[test_date, ticker]
actual_open = raw_open.loc[test_date, ticker]
prev_close = raw_close.iloc[raw_close.index.get_loc(test_date) - 1][ticker]

signal      = trading_signal(prev_close, actual_open, pred_price)
trade_ret   = compute_trade_return(signal, actual_open, actual_close)

In [140]:
signal, trade_ret

('SELL', np.float64(-0.03165829145728656))

In [147]:
df_smoothed[df_smoothed.index >= (pd.to_datetime('2022-01-03') - pd.Timedelta(days=WINDOW_SIZE))]

ticker,HALO
timestamp,
2021-12-06,32.692
2021-12-07,32.586
2021-12-08,32.800
2021-12-09,32.528
2021-12-10,32.948
...,...
2022-03-25,37.842
2022-03-28,38.236
2022-03-29,38.774


In [150]:
len(df_smoothed[df_smoothed.index >= (pd.to_datetime('2022-01-03') - pd.Timedelta(days=WINDOW_SIZE))])

81

In [152]:
train_data

ticker,HALO
timestamp,
2022-03-01,35.130
2022-03-02,35.582
2022-03-03,35.650
2022-03-04,35.566
2022-03-07,35.422
2022-03-08,35.150
2022-03-09,35.094
2022-03-10,35.144
2022-03-11,35.066


In [153]:
test_date

'2022-03-31'

In [151]:
# cool, now we have to move the window and retrain the model for each test day in the test window, 
# then compute signals and returns for each day.

for i, test_date in enumerate(pd.date_range(TEST_START, TEST_END)):
    test_date = test_date.strftime("%Y-%m-%d")
    print(f"Testing on {test_date}...")
    
    # Prepare training data up to the day before the test date
    train_data = df_smoothed[df_smoothed.index < test_date]
    train_data = train_data[train_data.index >= (pd.to_datetime(test_date) - pd.Timedelta(days=WINDOW_SIZE))]
    
    if len(train_data) < WINDOW_SIZE:
        print("Not enough training data, skipping this date.")
        print(train_data)
        continue
    
    # Scale
    scaled = scaler.fit_transform(train_data.values.reshape(-1, 1)).flatten()
    
    # Build sequences
    X, y = build_sequences(scaled, WINDOW_SIZE)
    X = X[..., np.newaxis]
    
    # Train model
    tf.keras.backend.clear_session()
    model = build_model(WINDOW_SIZE)
    model.fit(X, y,
              epochs=EPOCHS,
              batch_size=BATCH_SIZE,
              callbacks=[early_stop],
              verbose=0)
    
    # Predict
    last_window = scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
    pred_scaled = model.predict(last_window, verbose=0)[0, 0]
    pred_price  = scaler.inverse_transform([[pred_scaled]])[0, 0]
    
    # Compute signal and return
    actual_close = raw_close.loc[test_date, ticker]
    actual_open = raw_open.loc[test_date, ticker]
    prev_close = raw_close.iloc[raw_close.index.get_loc(test_date) - 1][ticker]

    signal      = trading_signal(prev_close, actual_open, pred_price)
    trade_ret   = compute_trade_return(signal, actual_open, actual_close)
    
    print(f"Predicted: {pred_price:.2f}, Actual Open: {actual_open:.2f}, Actual Close: {actual_close:.2f}, Signal: {signal}, Return: {trade_ret:.4f}\n")

Testing on 2022-01-03...
Not enough training data, skipping this date.
ticker        HALO
timestamp         
2021-12-06  32.692
2021-12-07  32.586
2021-12-08  32.800
2021-12-09  32.528
2021-12-10  32.948
2021-12-13  33.314
2021-12-14  33.772
2021-12-15  34.510
2021-12-16  35.274
2021-12-17  36.080
2021-12-20  36.782
2021-12-21  37.690
2021-12-22  38.210
2021-12-23  39.130
2021-12-27  39.498
2021-12-28  39.858
2021-12-29  40.080
2021-12-30  40.238
2021-12-31  40.232
Testing on 2022-01-04...
Not enough training data, skipping this date.
ticker        HALO
timestamp         
2021-12-06  32.692
2021-12-07  32.586
2021-12-08  32.800
2021-12-09  32.528
2021-12-10  32.948
2021-12-13  33.314
2021-12-14  33.772
2021-12-15  34.510
2021-12-16  35.274
2021-12-17  36.080
2021-12-20  36.782
2021-12-21  37.690
2021-12-22  38.210
2021-12-23  39.130
2021-12-27  39.498
2021-12-28  39.858
2021-12-29  40.080
2021-12-30  40.238
2021-12-31  40.232
2022-01-03  40.370
Testing on 2022-01-05...
Not enough train